# Foreign Whispers — Dubbing Studio (Colab)

Run every cell **top to bottom**.

**Before running** add to Colab Secrets (🔑 left sidebar):
- `HF_TOKEN` — HuggingFace token (pyannote model licence + Chatterbox download)
- `NGROK_TOKEN` — ngrok auth token (public studio URL)
- `GEMINI_API_KEY` — Gemini API key (translation re-ranking)

**YouTube cookies** are required — Colab IPs are flagged as bots. Export from your browser using the *Get cookies.txt LOCALLY* Chrome extension.

## 0 — Secrets & config

In [ ]:
import os
from google.colab import userdata

os.environ['MPLBACKEND']         = 'agg'
os.environ['FW_HF_TOKEN']        = userdata.get('HF_TOKEN')
os.environ['CHATTERBOX_API_URL'] = 'http://localhost:8020'
os.environ['FW_WHISPER_MODEL']   = 'base'
os.environ['GEMINI_API_KEY']     = userdata.get('GEMINI_API_KEY')

NGROK_TOKEN = userdata.get('NGROK_TOKEN')
VIDEO_ID    = 'GYQ5yGV_-Oc'
CONFIG      = 'c-fb1074a'
VIDEO_TITLE = 'Strait of Hormuz disruption threatens to shake global economy'
BASE        = '/content/foreign-whispers/pipeline_data/api'

print('HF_TOKEN set:     ', bool(os.environ.get('FW_HF_TOKEN')))
print('GEMINI_API_KEY set:', bool(os.environ.get('GEMINI_API_KEY')))
print('Config ready.')

## 0b — YouTube cookies

Colab IPs are flagged by YouTube — yt-dlp needs your browser cookies.

1. Install [Get cookies.txt LOCALLY](https://chrome.google.com/webstore/detail/get-cookiestxt-locally/cclelndahbckbenkjhflpdbgdldlbecc)
2. Go to `youtube.com` while logged in
3. Click the extension → export as `youtube.com_cookies.txt`
4. Upload that file in the cell below

Cookies expire periodically — re-export if downloads start failing again.

In [ ]:
import shutil, pathlib
from google.colab import files

cookies_dest = pathlib.Path('/content/youtube_cookies.txt')
if cookies_dest.exists():
    print('\u2705 Cookies already present')
else:
    print('Upload youtube.com_cookies.txt...')
    uploaded = files.upload()
    if uploaded:
        shutil.move(list(uploaded.keys())[0], str(cookies_dest))
        print('\u2705 Cookies saved')
    else:
        print('\u26a0\ufe0f No file uploaded \u2014 downloads may fail')

## 1 — System dependencies (~1 min)

In [ ]:
%%bash
apt-get install -y ffmpeg rubberband-cli imagemagick curl unzip \
    fonts-dejavu-core -q
curl -fsSL https://deno.land/install.sh | DENO_INSTALL=/usr/local sh
sed -i 's/rights="none" pattern="@\*"/rights="read|write" pattern="@*"/' \
    /etc/ImageMagick-6/policy.xml 2>/dev/null; true
# Verify rubberband binary is available for pyrubberband (TTS alignment)
which rubberband && echo "rubberband OK" || echo "ERROR: rubberband not found"
echo 'System deps done.'

## 2 — Clone repos & install deps (~5 min)

In [ ]:
%%bash
git clone https://github.com/aakashk99/foreign-whispers.git /content/foreign-whispers
cd /content/foreign-whispers
pip install uv -q
uv sync
# Install the 'alignment' dependency group which contains pyannote.audio
uv sync --group alignment
# Fix torchaudio: lockfile pins cu128 but Colab T4 has cu121.
# The cu128 C extension fails silently on cu121, leaving AudioMetaData undefined.
# --no-config bypasses pyproject.toml's find-links = cu128 constraint.
uv pip install torchaudio \
    --reinstall-package torchaudio \
    --index https://download.pytorch.org/whl/cu121 \
    --no-deps --no-config --python .venv/bin/python
# soundfile is required by torchaudio for audio backend support in pyannote
# Remove torchcodec from the venv so torchaudio falls back to soundfile.
# torchcodec installs fine but libtorchcodec.so fails to load on Colab T4.
find .venv -path "*torchcodec*" -exec rm -rf {} + 2>/dev/null; true
# Install soundfile into the venv (needed by torchaudio as audio backend)
pip install soundfile --target .venv/lib/python3.11/site-packages -q
mkdir -p pipeline_data/api/speakers
echo 'foreign-whispers ready.'

In [ ]:
from pathlib import Path

diarize_py = Path('/content/foreign-whispers/foreign_whispers/diarization.py')
src = diarize_py.read_text()

# Strip all previous patches — keep only the base module code
for marker in ['"""Speaker diarization', 'import logging']:
    idx = src.find(marker)
    if idx != -1:
        src = src[idx:]
        break

# Complete torchaudio replacement patch.
# Covers every torchaudio attribute pyannote.audio calls directly:
#   load(), info(), AudioMetaData, list_audio_backends()
# All imports captured as DEFAULT ARGS so 'del' at the end is safe.
PATCH = (
    '# \u2500\u2500 torchaudio patch \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n'
    '# torchaudio 2.9+ always uses torchcodec (not in Colab). Replace the 4\n'
    '# functions pyannote calls with soundfile implementations.\n'
    'try:\n'
    '    import torchaudio as _ta, collections as _col\n'
    '    import torch as _torch, numpy as _np, soundfile as _sf\n'
    '    _AM = _col.namedtuple("AudioMetaData",\n'
    '        ["sample_rate","num_frames","num_channels","bits_per_sample","encoding"])\n'
    '    def _load_sf(\n'
    '        uri, frame_offset=0, num_frames=-1, normalize=True,\n'
    '        channels_first=True, format=None, buffer_size=4096, backend=None,\n'
    '        _sf=_sf, _torch=_torch, _np=_np,\n'
    '    ):\n'
    '        data, sr = _sf.read(uri, start=frame_offset,\n'
    '            frames=num_frames if num_frames > 0 else -1,\n'
    '            dtype="float32", always_2d=True)\n'
    '        return _torch.from_numpy(_np.ascontiguousarray(\n'
    '            data.T if channels_first else data)), sr\n'
    '    def _info_sf(uri, backend=None, _sf=_sf, _AM=_AM):\n'
    '        i = _sf.info(uri)\n'
    '        return _AM(sample_rate=i.samplerate, num_frames=i.frames,\n'
    '            num_channels=i.channels, bits_per_sample=16, encoding="PCM_S")\n'
    '    _ta.load = _load_sf\n'
    '    _ta.info = _info_sf\n'
    '    _ta.AudioMetaData = _AM\n'
    '    if not hasattr(_ta, "list_audio_backends"):\n'
    '        _ta.list_audio_backends = lambda: ["soundfile"]\n'
    '    del _ta, _col, _torch, _np, _sf, _AM\n'
    'except Exception as _e:\n'
    '    import logging\n'
    '    logging.getLogger(__name__).warning("torchaudio patch failed: %s", _e)\n'
    '# \u2500\u2500 torch.load patch \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n'
    '# PyTorch 2.6 defaults weights_only=True; pyannote checkpoints need False.\n'
    'try:\n'
    '    import torch as _torch\n'
    '    def _ptl(*a, _orig=_torch.load, **kw):\n'
    '        kw["weights_only"] = False\n'
    '        return _orig(*a, **kw)\n'
    '    _torch.load = _ptl; del _torch\n'
    'except Exception: pass\n'
    '# \u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\u2500\n'
    '\n'
)

diarize_py.write_text(PATCH + src)
content = diarize_py.read_text()
print('\u2705 diarization.py patched')
print('  torchaudio.load:', '_load_sf' in content)
print('  torchaudio.info:', '_info_sf' in content)
print('  AudioMetaData:', '_AM' in content)
print('  list_audio_backends:', 'list_audio_backends' in content)
print('  torch.load:', 'weights_only' in content)
print('  default arg capture (no closure bug):', '_sf=_sf' in content)


In [ ]:
from pathlib import Path

checks = [
    ('tts.py: no speaker_wav=resolved_wav',
     'speaker_wav=resolved_wav' not in
     Path('/content/foreign-whispers/api/src/routers/tts.py').read_text()),
    ('tts_service.py: no speaker_wav param',
     'speaker_wav: str | None = None' not in
     Path('/content/foreign-whispers/api/src/services/tts_service.py').read_text()),
    ('download_engine.py: YT_COOKIES_FILE support',
     'YT_COOKIES_FILE' in
     Path('/content/foreign-whispers/api/src/services/download_engine.py').read_text()),
    ('reranking.py: uses Gemini',
     'generativelanguage.googleapis.com' in
     Path('/content/foreign-whispers/foreign_whispers/reranking.py').read_text()),
]
all_ok = True
for label, ok in checks:
    print(f'  {chr(10003) if ok else chr(10007)} {label}')
    if not ok: all_ok = False
print('\n\u2705 All checks passed' if all_ok
      else '\n\u274c Some checks FAILED \u2014 re-push the fixed files')

In [ ]:
%%bash
git clone https://github.com/speaches-ai/speaches.git /content/speaches
cd /content/speaches && uv python install && uv venv && uv sync
echo 'speaches ready.'

In [ ]:
%%bash
git clone https://github.com/travisvn/chatterbox-tts-api.git /content/chatterbox-tts-api
cd /content/chatterbox-tts-api
# Try python3 -m venv first; fall back to virtualenv if it fails.
# python3.12-venv from the deadsnakes PPA times out in Colab — virtualenv
# is a pure-Python alternative that works without any system package.
python3 -m venv .venv 2>/dev/null || (pip install virtualenv -q && python3 -m virtualenv .venv -q)
.venv/bin/pip install --upgrade pip setuptools wheel -q
.venv/bin/pip install fastapi 'uvicorn[standard]' python-multipart sse-starlette -q
test -f .venv/bin/pip && echo 'venv OK' || echo 'ERROR: venv not created'
echo 'chatterbox-tts-api ready.'

In [ ]:
import subprocess, pathlib

venv_pip = pathlib.Path('/content/chatterbox-tts-api/.venv/bin/pip')

if not venv_pip.exists():
    print('Venv missing \u2014 recreating with virtualenv...')
    subprocess.run(['pip', 'install', 'virtualenv', '-q'], check=True)
    subprocess.run(['python3', '-m', 'virtualenv', '.venv', '-q'],
        cwd='/content/chatterbox-tts-api', check=True)
    subprocess.run([str(venv_pip), 'install', '--upgrade', 'pip', 'setuptools', 'wheel', '-q'],
        cwd='/content/chatterbox-tts-api', check=True)
    subprocess.run([str(venv_pip), 'install',
        'fastapi', 'uvicorn[standard]', 'python-multipart', 'sse-starlette', '-q'],
        cwd='/content/chatterbox-tts-api', check=True)
    print('\u2705 Venv recreated')

print('Installing chatterbox-tts with cu121 torch...')
subprocess.run([
    str(venv_pip), 'install', '--force-reinstall', 'chatterbox-tts',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121',
], check=True)
subprocess.run([str(venv_pip), 'install', 'peft', 'resemble-perth'], check=True)
print('\u2705 chatterbox-tts installed.')

In [ ]:
from pathlib import Path

tts_py = Path('/content/chatterbox-tts-api/.venv/lib/python3.12/site-packages/chatterbox/tts.py')
if not tts_py.exists():
    print('\u26a0\ufe0f chatterbox/tts.py not found \u2014 run torch install cell first')
else:
    src = tts_py.read_text()
    if '_NoopWatermarker' not in src:
        src = src.replace(
            'self.watermarker = perth.PerthImplicitWatermarker()',
            'try:\n            self.watermarker = perth.PerthImplicitWatermarker()\n'
            '        except (TypeError, AttributeError):\n'
            '            class _NoopWatermarker:\n'
            '                def apply_watermark(self, audio, *a, **kw): return audio\n'
            '                def __call__(self, *a, **kw): return a[0] if a else None\n'
            '            self.watermarker = _NoopWatermarker()'
        )
        tts_py.write_text(src)
        print('\u2705 Patched chatterbox/tts.py with noop watermarker fallback')
    else:
        print('\u2705 Already patched')

## 3 — Build the Next.js frontend (~1 min)

`API_URL` must be set **at build time** so the Next.js proxy rewrites point at
FastAPI (port 8090). Without this, `/api/*` requests go to Jupyter Server.

In [ ]:
import subprocess, os

env = {**os.environ, 'API_URL': 'http://localhost:8090'}
subprocess.run(['npm', 'install'],
    cwd='/content/foreign-whispers/frontend', env=env,
    check=True, capture_output=True)
r = subprocess.run(['npx', 'next', 'build'],
    cwd='/content/foreign-whispers/frontend', env=env,
    capture_output=True, text=True)
print(r.stdout[-600:] or r.stderr[-400:])
print('Build exit code:', r.returncode)

## 4 — Start all services

- **STT** — speaches via `%%bash --bg`
- **Chatterbox** — venv uvicorn with HF tokens so model weights download
- **FastAPI** — `subprocess.Popen` so all env vars are inherited
- **Frontend** — `node .next/standalone/server.js` (`npx next start` breaks with `output:standalone`)

In [ ]:
%%bash --bg
cd /content/speaches && source .venv/bin/activate
MPLBACKEND=agg uvicorn --factory --host 0.0.0.0 --port 8000 \
    speaches.main:create_app >> /tmp/stt.log 2>&1

In [ ]:
import subprocess, os

hf_token = os.environ.get('FW_HF_TOKEN', '')
chatterbox_proc = subprocess.Popen(
    ['/content/chatterbox-tts-api/.venv/bin/uvicorn',
     'app.main:app', '--host', '0.0.0.0', '--port', '8020'],
    cwd='/content/chatterbox-tts-api',
    env={**os.environ,
         'DEVICE':                 'cuda',
         'USE_MULTILINGUAL_MODEL': 'false',
         'PORT':                   '8020',
         'HF_TOKEN':               hf_token,
         'HUGGING_FACE_HUB_TOKEN': hf_token},
    stdout=open('/tmp/tts.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('Chatterbox started (PID', chatterbox_proc.pid, ')', '| HF_TOKEN:', bool(hf_token))

In [ ]:
import subprocess, os

# Ensure rubberband-cli is installed — pyrubberband needs it for TTS alignment
r = subprocess.run(['which', 'rubberband'], capture_output=True)
if r.returncode != 0:
    print('rubberband not found — installing...')
    subprocess.run(['apt-get', 'install', '-y', 'rubberband-cli', '-q'], check=True)
    subprocess.run(['which', 'rubberband'], check=True)
    print('\u2705 rubberband installed')
else:
    print('\u2705 rubberband found:', r.stdout.decode().strip())

import subprocess, os
from pathlib import Path

# Re-load token from secrets if missing
if not os.environ.get('FW_HF_TOKEN'):
    try:
        from google.colab import userdata
        os.environ['FW_HF_TOKEN'] = userdata.get('HF_TOKEN')
        print('Re-loaded FW_HF_TOKEN from Colab secrets')
    except Exception as e:
        print(f'\u26a0\ufe0f Could not load HF_TOKEN: {e}')

hf_token = os.environ.get('FW_HF_TOKEN', '')
print('FW_HF_TOKEN set:', bool(hf_token))

# Clear ALL .pyc caches (project + venv) so patches always take effect
for _d in ['/content/foreign-whispers', '/content/foreign-whispers/.venv']:
    subprocess.run(['find', _d, '-name', '*.pyc', '-delete'], capture_output=True)
print('\u2705 Bytecode caches cleared')

api_proc = subprocess.Popen(
    ['uv', 'run', 'uvicorn', 'api.src.main:app',
     '--host', '0.0.0.0', '--port', '8090'],
    cwd='/content/foreign-whispers',
    env={**os.environ,
         'CHATTERBOX_API_URL': 'http://localhost:8020',
         'FW_WHISPER_MODEL':   'base',
         'MPLBACKEND':         'agg',
         'YT_COOKIES_FILE':    '/content/youtube_cookies.txt',
         'FW_HF_TOKEN':        hf_token,
         'TORCHAUDIO_USE_BACKEND': 'soundfile'},
    stdout=open('/tmp/api.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('FastAPI started (PID', api_proc.pid, ')')

In [ ]:
import subprocess, os

fe_proc = subprocess.Popen(
    ['node', '.next/standalone/server.js'],
    cwd='/content/foreign-whispers/frontend',
    env={**os.environ,
         'API_URL':  'http://localhost:8090',
         'HOSTNAME': '0.0.0.0',
         'PORT':     '8501'},
    stdout=open('/tmp/frontend.log', 'w'),
    stderr=subprocess.STDOUT,
)
print('Frontend started (PID', fe_proc.pid, ')')

## 5 — Wait for services (~3–5 min)

**Do not proceed until Chatterbox shows ✅ model loaded.**

In [ ]:
import requests, time

services = {
    'STT (speaches)':   'http://localhost:8000/health',
    'TTS (chatterbox)': 'http://localhost:8020/health',
    'FastAPI':          'http://localhost:8090/healthz',
    'Frontend':         'http://localhost:8501',
}
for name, url in services.items():
    for attempt in range(24):
        try:
            if requests.get(url, timeout=3).ok:
                print(f'\u2705 {name} is up'); break
        except Exception:
            pass
        print(f'\u23f3 Waiting for {name}... ({attempt+1}/24)'); time.sleep(10)
    else:
        print(f'\u274c {name} failed \u2014 check logs below')

In [ ]:
import requests, time

print('Waiting for Chatterbox model to fully load (3-5 min)...')
for attempt in range(36):
    try:
        r = requests.get('http://localhost:8020/health', timeout=3).json()
        if r.get('model_loaded'):
            print('\u2705 Chatterbox model loaded and ready'); break
    except Exception:
        pass
    print(f'  \u23f3 Still loading... ({attempt+1}/36)'); time.sleep(10)
else:
    print('\u274c Model never loaded \u2014 run Section 5b')

In [ ]:
%%bash
echo '=== STT ===' && tail -15 /tmp/stt.log
echo '=== TTS ===' && tail -20 /tmp/tts.log
echo '=== API ===' && tail -15 /tmp/api.log
echo '=== Frontend ===' && tail -10 /tmp/frontend.log

## 5b — Chatterbox recovery (only run if Section 5 showed ❌)

In [ ]:
import subprocess, os, time, requests, pathlib

print('Killing existing Chatterbox process...')
subprocess.run(['fuser', '-k', '8020/tcp'], capture_output=True)
time.sleep(3)

hf_token = os.environ.get('FW_HF_TOKEN', '')
venv_pip  = pathlib.Path('/content/chatterbox-tts-api/.venv/bin/pip')

if not venv_pip.exists():
    print('Recreating venv with virtualenv...')
    subprocess.run(['pip', 'install', 'virtualenv', '-q'], check=True)
    subprocess.run(['python3', '-m', 'virtualenv', '.venv', '-q'],
        cwd='/content/chatterbox-tts-api', check=True)
    subprocess.run([str(venv_pip), 'install', '--upgrade', 'pip', 'setuptools', 'wheel'],
        cwd='/content/chatterbox-tts-api', check=True)
    subprocess.run([str(venv_pip), 'install',
        'fastapi', 'uvicorn[standard]', 'python-multipart', 'sse-starlette'],
        cwd='/content/chatterbox-tts-api', check=True)

print('Reinstalling chatterbox-tts...')
subprocess.run([str(venv_pip), 'install', '--force-reinstall', 'chatterbox-tts',
    '--extra-index-url', 'https://download.pytorch.org/whl/cu121'],
    check=True, capture_output=True)
subprocess.run([str(venv_pip), 'install', 'peft', 'resemble-perth'],
    check=True, capture_output=True)

# Re-apply watermarker patch
tts_py = pathlib.Path('/content/chatterbox-tts-api/.venv/lib/python3.12/site-packages/chatterbox/tts.py')
if tts_py.exists():
    src = tts_py.read_text()
    if '_NoopWatermarker' not in src:
        src = src.replace(
            'self.watermarker = perth.PerthImplicitWatermarker()',
            'try:\n            self.watermarker = perth.PerthImplicitWatermarker()\n'
            '        except (TypeError, AttributeError):\n'
            '            class _NoopWatermarker:\n'
            '                def apply_watermark(self, audio, *a, **kw): return audio\n'
            '                def __call__(self, *a, **kw): return a[0] if a else None\n'
            '            self.watermarker = _NoopWatermarker()'
        )
        tts_py.write_text(src)
    print('\u2705 Watermarker patch applied')

chatterbox_proc = subprocess.Popen(
    ['/content/chatterbox-tts-api/.venv/bin/uvicorn',
     'app.main:app', '--host', '0.0.0.0', '--port', '8020'],
    cwd='/content/chatterbox-tts-api',
    env={**os.environ,
         'DEVICE': 'cuda', 'USE_MULTILINGUAL_MODEL': 'false', 'PORT': '8020',
         'HF_TOKEN': hf_token, 'HUGGING_FACE_HUB_TOKEN': hf_token},
    stdout=open('/tmp/tts.log', 'w'), stderr=subprocess.STDOUT,
)
print('Restarted PID', chatterbox_proc.pid); time.sleep(5)
print('Still running:', chatterbox_proc.poll() is None)
print(open('/tmp/tts.log').read()[-400:])

print('\nWaiting for model...')
for attempt in range(36):
    time.sleep(10)
    if chatterbox_proc.poll() is not None:
        print('Process exited. Full log:'); print(open('/tmp/tts.log').read()); break
    try:
        r = requests.get('http://localhost:8020/health', timeout=3).json()
        print(f'  attempt {attempt+1}: model_loaded={r.get("model_loaded")}')
        if r.get('model_loaded'): print('\u2705 Ready'); break
    except Exception as e:
        print(f'  attempt {attempt+1}: {e}')
else:
    print('\u274c Still failing. Full log:'); print(open('/tmp/tts.log').read())

## 6 — Expose studio via ngrok

Open the printed URL — **not** `localhost:8501`.

In [ ]:
!pip install pyngrok -q
from pyngrok import ngrok
ngrok.set_auth_token(NGROK_TOKEN)
public_url = ngrok.connect(8501)
print('\U0001f3ac Dubbing Studio:', public_url)

## 7 — Run the end-to-end pipeline via nbconvert

In [ ]:
%%bash
sed -i 's|http://localhost:8080|http://localhost:8090|g' \
  /content/foreign-whispers/notebooks/pipeline_end_to_end/pipeline_end_to_end.ipynb
echo "Patched: $(grep -c 8090 \
  /content/foreign-whispers/notebooks/pipeline_end_to_end/pipeline_end_to_end.ipynb) occurrences"

In [ ]:
import subprocess, os

result = subprocess.run(
    ['uv', 'run', 'jupyter', 'nbconvert', '--to', 'notebook', '--execute',
     'notebooks/pipeline_end_to_end/pipeline_end_to_end.ipynb',
     '--output', '/content/pipeline_end_to_end_output.ipynb',
     '--ExecutePreprocessor.timeout=600'],
    cwd='/content/foreign-whispers',
    env=os.environ.copy(),
    text=True, capture_output=False,
)
print('\u2705 Done' if result.returncode == 0
      else f'\u274c Failed (code {result.returncode}) \u2014 use Section 7b instead')

## 7b — Manual bypass: process all four videos

Use this section **instead of Section 7** if nbconvert fails or YouTube blocks downloads.

Upload each MP4 directly (download from YouTube in your browser), then run the full
pipeline for all four videos including diarization. Dubbed videos are downloaded at the end.

In [ ]:
import shutil, pathlib
from google.colab import files

VIDEOS = [
    ('GYQ5yGV_-Oc',  'Strait of Hormuz disruption threatens to shake global economy'),
    ('6O3HLPWatuU',  'Alysa Liu: The 60 Minutes Interview'),
    ('DLeBquj8LKI',  'Rob Reiner: The 60 Minutes Interview'),
    ('IiBKsv-D64M',  'Military Drones: 60 Minutes Full Episodes'),
]

videos_dir = pathlib.Path(f'{BASE}/videos')
videos_dir.mkdir(parents=True, exist_ok=True)

print('Checking which videos need uploading...')
for vid_id, title in VIDEOS:
    dest = videos_dir / f'{title}.mp4'
    if dest.exists():
        print(f'  \u2705 Already present: {title[:50]}')
    else:
        print(f'  \u2197 Upload MP4 for: {title}')
        print(f'     YouTube URL: https://www.youtube.com/watch?v={vid_id}')
        uploaded = files.upload()
        if uploaded:
            fname = list(uploaded.keys())[0]
            shutil.move(fname, str(dest))
            print(f'  \u2705 Saved: {dest.name}')
        else:
            print(f'  \u274c Skipped \u2014 re-run this cell to retry')

print('\nUpload check complete.')
for vid_id, title in VIDEOS:
    dest = videos_dir / f'{title}.mp4'
    status = chr(10003) if dest.exists() else chr(10007)
    print(f'  {status} {title[:55]}')

In [ ]:
import requests, time, pathlib, subprocess

API = 'http://localhost:8090'

VIDEOS = [
    ('GYQ5yGV_-Oc',  'Strait of Hormuz disruption threatens to shake global economy',  'c-fb1074a'),
    ('6O3HLPWatuU',  'Alysa Liu: The 60 Minutes Interview',                            'c-fb1074a'),
    ('DLeBquj8LKI',  'Rob Reiner: The 60 Minutes Interview',                           'c-fb1074a'),
    ('IiBKsv-D64M',  'Military Drones: 60 Minutes Full Episodes',                      'c-fb1074a'),
]

# Check whether diarization is enabled in studio settings
# (set USE_DIARIZATION = True to enable speaker diarization for all videos)
USE_DIARIZATION = False  # set to True if pyannote licence is accepted

def run_stage(name, url, params=None, timeout=600):
    r = requests.post(url, params=params, timeout=timeout)
    if not r.ok:
        return False, f'{r.status_code}: {r.text[:200]}'
    data = r.json() if r.headers.get('content-type','').startswith('application/json') else {}
    skipped = data.get('skipped', False)
    return True, 'skipped' if skipped else 'done'

results = {}

for vid_id, title, config in VIDEOS:
    print(f'\n{"="*60}')
    print(f'Processing: {title[:55]}')
    print(f'{"="*60}')

    # Skip if source video not uploaded
    src_mp4 = pathlib.Path(f'{BASE}/videos/{title}.mp4')
    if not src_mp4.exists():
        print(f'  \u274c Source MP4 not found \u2014 skipping (upload in cell above)')
        results[vid_id] = 'skipped (no source video)'
        continue

    stages = [
        ('Transcribe', f'{API}/api/transcribe/{vid_id}', None,                             300),
        ('Translate',  f'{API}/api/translate/{vid_id}',  None,                             120),
        ('TTS',        f'{API}/api/tts/{vid_id}',        {'config': config, 'alignment': 'true'}, 600),
        ('Stitch',     f'{API}/api/stitch/{vid_id}',     {'config': config},               120),
    ]

    # Insert diarize stage after transcribe if enabled
    if USE_DIARIZATION:
        stages.insert(1, ('Diarize', f'{API}/api/diarize/{vid_id}', None, 300))

    all_ok = True
    for stage_name, url, params, timeout in stages:
        ok, status = run_stage(stage_name, url, params, timeout)
        icon = chr(9197) if status == 'skipped' else chr(10003) if ok else chr(10007)
        print(f'  {icon} {stage_name}: {status}')
        if not ok:
            all_ok = False
            print(f'     Stopping pipeline for this video.')
            results[vid_id] = f'failed at {stage_name}: {status}'
            break

    if all_ok:
        results[vid_id] = 'complete'

# Summary
print(f'\n{"="*60}')
print('PIPELINE SUMMARY')
print(f'{"="*60}')
for vid_id, title, config in VIDEOS:
    status = results.get(vid_id, 'not run')
    icon = chr(10003) if status == 'complete' else chr(10007)
    print(f'  {icon} [{vid_id}] {title[:50]}: {status}')
print('\nRun Section 8b below to stitch and download all completed videos.')

## 8 — Stitch & download dubbed video

Calls ffmpeg directly with explicit stream mapping to guarantee both video and audio tracks are present.

In [ ]:
import subprocess, pathlib

src_mp4 = f'{BASE}/videos/{VIDEO_TITLE}.mp4'
src_wav = f'{BASE}/tts_audio/chatterbox/{CONFIG}/{VIDEO_TITLE}.wav'
out_mp4 = f'{BASE}/dubbed_videos/{CONFIG}/{VIDEO_TITLE}.mp4'

print('Source files:')
for f in [src_mp4, src_wav]:
    e = pathlib.Path(f).exists()
    print(f'  {chr(10003) if e else chr(10007)} {pathlib.Path(f).name}')

pathlib.Path(out_mp4).parent.mkdir(parents=True, exist_ok=True)
r = subprocess.run([
    'ffmpeg', '-y', '-i', src_mp4, '-i', src_wav,
    '-map', '0:v:0', '-map', '1:a:0',
    '-c:v', 'copy', '-c:a', 'aac', '-shortest', out_mp4,
], capture_output=True, text=True)

if r.returncode == 0:
    print(f'\u2705 {pathlib.Path(out_mp4).stat().st_size/1e6:.1f} MB')
else:
    print('\u274c ffmpeg error:'); print(r.stderr[-500:])

In [ ]:
from google.colab import files
import pathlib

out = f'{BASE}/dubbed_videos/{CONFIG}/{VIDEO_TITLE}.mp4'
if pathlib.Path(out).exists():
    files.download(out)
else:
    print('\u274c Not found \u2014 run stitch cell first')

## 8b — Stitch & download all four dubbed videos

Stitches each completed video with its dubbed audio and triggers a browser download. Skips any video whose TTS audio doesn't exist yet.

In [ ]:
import subprocess, pathlib
from google.colab import files

VIDEOS = [
    ('GYQ5yGV_-Oc',  'Strait of Hormuz disruption threatens to shake global economy',  'c-fb1074a'),
    ('6O3HLPWatuU',  'Alysa Liu: The 60 Minutes Interview',                            'c-fb1074a'),
    ('DLeBquj8LKI',  'Rob Reiner: The 60 Minutes Interview',                           'c-fb1074a'),
    ('IiBKsv-D64M',  'Military Drones: 60 Minutes Full Episodes',                      'c-fb1074a'),
]

for vid_id, title, config in VIDEOS:
    src_mp4 = pathlib.Path(f'{BASE}/videos/{title}.mp4')
    src_wav = pathlib.Path(f'{BASE}/tts_audio/chatterbox/{config}/{title}.wav')
    out_mp4 = pathlib.Path(f'{BASE}/dubbed_videos/{config}/{title}.mp4')

    print(f'\n[{vid_id}] {title[:50]}')

    if not src_mp4.exists():
        print(f'  \u274c Source MP4 missing \u2014 skipping'); continue
    if not src_wav.exists():
        print(f'  \u274c TTS audio missing \u2014 run pipeline cell first'); continue

    out_mp4.parent.mkdir(parents=True, exist_ok=True)

    r = subprocess.run([
        'ffmpeg', '-y',
        '-i', str(src_mp4),
        '-i', str(src_wav),
        '-map', '0:v:0',
        '-map', '1:a:0',
        '-c:v', 'copy',
        '-c:a', 'aac',
        '-shortest',
        str(out_mp4),
    ], capture_output=True, text=True)

    if r.returncode == 0:
        size_mb = out_mp4.stat().st_size / 1e6
        print(f'  \u2705 Stitched ({size_mb:.1f} MB) \u2014 downloading...')
        files.download(str(out_mp4))
    else:
        print(f'  \u274c ffmpeg error:')
        print('   ', r.stderr[-300:])

print('\n\u2705 All done.')

## 9 — Re-run TTS if audio was sparse

Only run if the dubbed video has many silent segments (TTS ran before Chatterbox loaded).

In [ ]:
import pathlib

for path in [
    f'{BASE}/tts_audio/chatterbox/{CONFIG}/{VIDEO_TITLE}.wav',
    f'{BASE}/tts_audio/chatterbox/{CONFIG}/{VIDEO_TITLE}.align.json',
    f'{BASE}/dubbed_videos/{CONFIG}/{VIDEO_TITLE}.mp4',
]:
    p = pathlib.Path(path)
    if p.exists(): p.unlink(); print(f'Deleted: {p.name}')
    else: print(f'Not found (ok): {p.name}')
print('Cache cleared.')

In [ ]:
import requests

r = requests.get('http://localhost:8020/health', timeout=5).json()
if not r.get('model_loaded'):
    raise RuntimeError('Chatterbox not ready \u2014 run Section 5b first')
print('\u2705 Chatterbox ready. Synthesising...')

resp = requests.post(f'http://localhost:8090/api/tts/{VIDEO_ID}',
    params={'config': CONFIG, 'alignment': 'true'}, timeout=600)
print('Status:', resp.status_code)
if resp.ok: print('\u2705 TTS done:', resp.json().get('audio_path'))
else: print('\u274c Error:', resp.text[:400])

In [ ]:
import subprocess, pathlib
from google.colab import files

src_mp4 = f'{BASE}/videos/{VIDEO_TITLE}.mp4'
src_wav = f'{BASE}/tts_audio/chatterbox/{CONFIG}/{VIDEO_TITLE}.wav'
out_mp4 = f'{BASE}/dubbed_videos/{CONFIG}/{VIDEO_TITLE}.mp4'
pathlib.Path(out_mp4).parent.mkdir(parents=True, exist_ok=True)
r = subprocess.run([
    'ffmpeg', '-y', '-i', src_mp4, '-i', src_wav,
    '-map', '0:v:0', '-map', '1:a:0',
    '-c:v', 'copy', '-c:a', 'aac', '-shortest', out_mp4,
], capture_output=True, text=True)
if r.returncode == 0:
    print(f'\u2705 {pathlib.Path(out_mp4).stat().st_size/1e6:.1f} MB')
    files.download(out_mp4)
else:
    print('\u274c', r.stderr[-500:])

## Troubleshooting

| Symptom | Cause | Fix |
|---------|-------|-----|
| `Jupyter Server` HTML error in studio | Frontend built without `API_URL` | Rerun Sections 3 & 4 |
| `localhost:8501` unreachable | Colab runs on Google's servers | Use the ngrok URL |
| ngrok tunnel expired | Free tunnels expire after ~2 hrs | Rerun ngrok cell |
| Chatterbox `ModuleNotFoundError: chatterbox` | System uvicorn used | Already fixed: venv uvicorn in Section 4 |
| Chatterbox `NoneType` / `AttributeError: PerthImplicitWatermarker` | perth broken on Py3.12 | Already fixed: noop watermarker patch |
| Chatterbox model never loads | Download failed / CUDA issue | Run Section 5b |
| `torchaudio` C extension errors (AudioMetaData, list_audio_backends) | CUDA build mismatch | Already fixed: CUDA detection + io.py patch in Section 2 |
| `pyannote.audio` not installed | In `alignment` dep group, skipped by `uv sync` | Already fixed: `uv sync --group alignment` |
| Diarize: `FW_HF_TOKEN not set` | API started without env vars | Already fixed: `subprocess.Popen` in Section 4 |
| Dubbed audio mostly silent | TTS ran before Chatterbox loaded | Run Section 9 |
| Output has audio but no video | ffmpeg dropped video stream | Run Section 8 stitch cell |
| YouTube bot-detection error | Cookies expired | Re-export cookies (Section 0b), restart API |
| `speaker_wav` unexpected kwarg | Stale `.pyc` bytecode | Already fixed: cache cleared in Section 4 |